# Mental Health Late Fusion Model

Notebook ini menggunakan:
- Deep Learning untuk data tabular
- NLP sederhana (TextVectorization + BiLSTM)
- Late Fusion Multi-Output Model
- TensorFlow Functional API
- Custom Layer


In [ ]:
# =========================================================
# 1. IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Embedding,
    Bidirectional,
    LSTM,
    TextVectorization,
    Layer
)

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [ ]:
# =========================================================
# 2. LOAD SAMPLE DATA
# =========================================================

tabular_df = pd.DataFrame({

    "Age": [22,25,30,28,35,21,40,19],
    "Gender": ["Male","Female","Male","Female","Male","Female","Male","Female"],
    "family_history": ["No","Yes","Yes","No","Yes","No","Yes","No"],
    "treatment": ["No","Yes","Yes","No","Yes","No","Yes","No"]

})

text_df = pd.DataFrame({

    "text": [

        "i feel anxious and worried",
        "i feel depressed and hopeless",
        "i am okay and happy",
        "i want to end my life",
        "panic attack and anxiety",
        "normal day and calm",
        "suicidal thoughts everyday",
        "feeling very depressed"

    ],

    "status": [

        "Anxiety",
        "Depression",
        "Normal",
        "Suicidal",
        "Anxiety",
        "Normal",
        "Suicidal",
        "Depression"

    ]

})

print(tabular_df.head())
print(text_df.head())


In [ ]:
# =========================================================
# 3. ENCODE TABULAR DATA
# =========================================================

gender_encoder = LabelEncoder()
family_encoder = LabelEncoder()
treatment_encoder = LabelEncoder()

tabular_df["Gender"] = gender_encoder.fit_transform(tabular_df["Gender"])
tabular_df["family_history"] = family_encoder.fit_transform(
    tabular_df["family_history"]
)

tabular_df["treatment"] = treatment_encoder.fit_transform(
    tabular_df["treatment"]
)

X_tab = tabular_df.drop("treatment", axis=1)
y_tab = tabular_df["treatment"]

scaler = StandardScaler()

X_tab = scaler.fit_transform(X_tab)


In [ ]:
# =========================================================
# 4. ENCODE TEXT LABEL
# =========================================================

text_encoder = LabelEncoder()

text_df["status"] = text_encoder.fit_transform(
    text_df["status"]
)

X_text = text_df["text"].values
y_text = text_df["status"].values

print(text_encoder.classes_)


In [ ]:
# =========================================================
# 5. TRAIN TEST SPLIT
# =========================================================

X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
    X_tab,
    y_tab,
    test_size=0.2,
    random_state=42
)

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text,
    y_text,
    test_size=0.2,
    random_state=42
)


In [ ]:
# =========================================================
# 6. TEXT VECTORIZATION
# =========================================================

VOCAB_SIZE = 5000
MAX_LEN = 50

vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN
)

vectorizer.adapt(X_text_train)


In [ ]:
# =========================================================
# 7. CUSTOM LAYER
# =========================================================

class AttentionLayer(Layer):

    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):

        self.W = self.add_weight(
            shape=(input_shape[-1], 1),
            initializer="random_normal",
            trainable=True
        )

    def call(self, inputs):

        attention = tf.nn.softmax(
            tf.matmul(inputs, self.W),
            axis=1
        )

        context = attention * inputs

        return tf.reduce_sum(context, axis=1)


In [ ]:
# =========================================================
# 8. TABULAR INPUT BRANCH
# =========================================================

tabular_input = Input(
    shape=(X_tab_train.shape[1],),
    name="tabular_input"
)

x1 = Dense(128, activation="relu")(tabular_input)

x1 = Dropout(0.3)(x1)

x1 = Dense(64, activation="relu")(x1)

x1 = Dense(32, activation="relu")(x1)

tabular_output = Dense(
    2,
    activation="softmax",
    name="tabular_output"
)(x1)


In [ ]:
# =========================================================
# 9. TEXT INPUT BRANCH
# =========================================================

text_input = Input(
    shape=(1,),
    dtype=tf.string,
    name="text_input"
)

vectorized_text = vectorizer(text_input)

x2 = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=128
)(vectorized_text)

x2 = Bidirectional(
    LSTM(
        64,
        return_sequences=True
    )
)(x2)

x2 = AttentionLayer()(x2)

x2 = Dense(64, activation="relu")(x2)

x2 = Dropout(0.3)(x2)

text_output = Dense(
    4,
    activation="softmax",
    name="text_output"
)(x2)


In [ ]:
# =========================================================
# 10. BUILD MODEL
# =========================================================

model = Model(

    inputs=[
        tabular_input,
        text_input
    ],

    outputs=[
        tabular_output,
        text_output
    ]

)

model.summary()


In [ ]:
# =========================================================
# 11. COMPILE MODEL
# =========================================================

model.compile(

    optimizer="adam",

    loss={
        "tabular_output": "sparse_categorical_crossentropy",
        "text_output": "sparse_categorical_crossentropy"
    },

    metrics={
        "tabular_output": ["accuracy"],
        "text_output": ["accuracy"]
    }

)


In [ ]:
# =========================================================
# 12. TRAIN MODEL
# =========================================================

history = model.fit(

    {
        "tabular_input": X_tab_train,
        "text_input": np.array(X_text_train).reshape(-1,1)
    },

    {
        "tabular_output": y_tab_train,
        "text_output": y_text_train
    },

    validation_data=(

        {
            "tabular_input": X_tab_test,
            "text_input": np.array(X_text_test).reshape(-1,1)
        },

        {
            "tabular_output": y_tab_test,
            "text_output": y_text_test
        }

    ),

    epochs=10,

    callbacks=[
        EarlyStopping(
            patience=3,
            restore_best_weights=True
        )
    ]

)


In [ ]:
# =========================================================
# 13. SAVE MODEL
# =========================================================

model.save("mental_health_late_fusion.keras")

print("Model berhasil disimpan!")


In [ ]:
# =========================================================
# 14. LOAD MODEL
# =========================================================

loaded_model = tf.keras.models.load_model(
    "mental_health_late_fusion.keras",
    custom_objects={
        "AttentionLayer": AttentionLayer
    }
)

print("Model berhasil di-load!")


In [ ]:
# =========================================================
# 15. INFERENCE
# =========================================================

sample_tabular = np.array([
    X_tab_test[0]
])

sample_text = np.array([
    "i feel anxious and depressed lately"
], dtype=object).reshape(-1,1)

result = loaded_model.predict({

    "tabular_input": sample_tabular,
    "text_input": sample_text

})

print(result)


In [ ]:
# =========================================================
# 16. LABEL RESULT
# =========================================================

tabular_pred = np.argmax(result[0][0])

text_pred = np.argmax(result[1][0])

tabular_label = ["No", "Yes"][tabular_pred]

text_labels = [
    "Anxiety",
    "Depression",
    "Normal",
    "Suicidal"
]

text_label = text_labels[text_pred]

print("Tabular Prediction :", tabular_label)
print("Text Prediction     :", text_label)
